In [34]:
!pip install -q langchain langgraph langchain-groq pypdf

In [35]:
import os
from google.colab import userdata

from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

import pypdf

In [36]:
#creando variable de entorno
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [37]:
# Cargar y leer el PDF
ruta_pdf = "/content/techshop-documentacion.pdf"

with open(ruta_pdf, "rb") as archivo:
    lector = pypdf.PdfReader(archivo)
    texto_documento = ""
    for pagina in lector.pages:
        texto_pagina = pagina.extract_text()
        if texto_pagina:
            # Limpiar espacios extraños y unir palabras separadas
            texto_pagina = texto_pagina.replace("\n", " ")
            texto_pagina = " ".join(texto_pagina.split())
            texto_documento += texto_pagina + "\n\n"

# Ver los primeros 500 caracteres para confirmar
print("PDF cargado correctamente. Vista previa:")
print(texto_documento[:500])
print(f"\n Longitud total del texto: {len(texto_documento)} caracteres")



PDF cargado correctamente. Vista previa:
TECHSHOP - DOCUMENTACIÓN OFICIAL Tienda Online de Productos Electrónicos Versión 2.4 - Marzo 2025 =========================================== 1. POLÍTICA DE REEMBOLSO Y DEVOLUCIONES =========================================== 1.1 Plazo de devolución Los clientes de TechShop tienen derecho a devolver cualquier producto dentro de los primeros 30 días corridos desde la fecha de recepción del pedido. Pasado ese plazo, no se aceptarán devoluciones por ningún motivo salvo defectos de fábrica cubiertos

 Longitud total del texto: 4788 caracteres


In [45]:
@tool
def buscar_en_documento(query: str) -> str:
    """
    Busca información relevante en el documento PDF de TechShop.
    """
    # Dividir en párrafos
    parrafos = texto_documento.split("\n")

    # Palabras clave (ignoramos palabras cortas)
    palabras = query.lower().split()
    palabras_clave = [p for p in palabras if len(p) > 3]

    if not palabras_clave:
        palabras_clave = palabras

    resultados = []

    # Buscar párrafos que contengan AL MENOS UNA palabra clave
    for i, parrafo in enumerate(parrafos):
        parrafo_lower = parrafo.lower()
        for palabra in palabras_clave:
            if palabra in parrafo_lower:
                # Agarrar contexto amplio
                inicio = max(0, i - 2)
                fin = min(len(parrafos), i + 3)
                contexto = "\n".join(parrafos[inicio:fin])
                resultados.append(contexto)
                break  # Ya encontré una, paso al siguiente párrafo

    if not resultados:
        return "No se encontró información en el documento."

    # Devolver hasta 5 fragmentos
    return "\n\n---\n\n".join(resultados[:5])

In [47]:
# Probar la herramienta directamente (sin el agente)
print("=== PRUEBA 1: devolución ===")
print(buscar_en_documento.invoke("devolución producto abierto empaque"))

print("\n=== PRUEBA 2: métodos de pago ===")
print(buscar_en_documento.invoke("métodos de pago"))

print("\n=== PRUEBA 3: envío express ===")
print(buscar_en_documento.invoke("envío express cuesta tarda"))

=== PRUEBA 1: devolución ===
TECHSHOP - DOCUMENTACIÓN OFICIAL Tienda Online de Productos Electrónicos Versión 2.4 - Marzo 2025 =========================================== 1. POLÍTICA DE REEMBOLSO Y DEVOLUCIONES =========================================== 1.1 Plazo de devolución Los clientes de TechShop tienen derecho a devolver cualquier producto dentro de los primeros 30 días corridos desde la fecha de recepción del pedido. Pasado ese plazo, no se aceptarán devoluciones por ningún motivo salvo defectos de fábrica cubiertos por la garantía. 1.2 Condiciones del producto Para que una devolución sea aceptada, el producto debe cumplir las siguientes condiciones: - Estar en su empaque original, sin daños ni alteraciones. - Incluir todos los accesorios, manuales y cables que venían con el producto. - No presentar signos de uso excesivo, golpes, rayaduras o derrames de líquidos. - Tener los sellos de seguridad intactos (si aplica). 1.3 Productos no retornables Los siguientes productos no pued

In [49]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
    api_key=os.environ["GROQ_API_KEY"]
)

print("✅ LLM de Groq inicializado correctamente")

✅ LLM de Groq inicializado correctamente


In [50]:
system_prompt = """
Eres un asistente virtual de TechShop, una tienda online de productos electrónicos.
Tu única fuente de verdad es el resultado de la herramienta 'buscar_en_documento'.

Reglas:
1. SIEMPRE usa la herramienta 'buscar_en_documento' para responder.
2. No inventes información que no esté en el documento.
3. Si el documento no contiene la respuesta, dilo claramente.
4. Responde en español, de forma clara, amigable y concisa.
5. Cuando uses información del documento, menciona en qué sección se encuentra.
6. Si el usuario pregunta algo fuera del alcance del documento, indica amablemente que solo podés responder sobre temas de TechShop.
"""

agente = create_react_agent(
    model=llm,
    tools=[buscar_en_documento],
    prompt=system_prompt
)

print("Agente creado correctamente")

Agente creado correctamente


/tmp/ipykernel_573/2373640616.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente = create_react_agent(


In [51]:
pregunta = "¿Cuál es el plazo para devolver un producto?"

resultado = agente.invoke({
    "messages": [("user", pregunta)]
})

respuesta_final = resultado["messages"][-1].content

print("Usuario:", pregunta)
print("Agente:", respuesta_final)

Usuario: ¿Cuál es el plazo para devolver un producto?
Agente: El plazo para devolver un producto es de 30 días corridos desde la fecha de recepción del pedido.


In [52]:
pregunta1 = "¿Puedo devolver un producto si ya abrí el empaque?"

resultado1 = agente.invoke({
    "messages": [("user", pregunta1)]
})

print("Usuario:", pregunta1)
print("Agente:", resultado1["messages"][-1].content)

Usuario: ¿Puedo devolver un producto si ya abrí el empaque?
Agente: Lo siento, pero según la documentación de TechShop, si ya abriste el empaque, no podrás devolver el producto. La condición para devolver un producto es que esté en su empaque original, sin daños ni alteraciones. Si ya abriste el empaque, el producto no cumple con esta condición y, por lo tanto, no se aceptará la devolución.


In [53]:
pregunta2 = "¿Qué métodos de pago aceptan en TechShop?"

resultado2 = agente.invoke({
    "messages": [("user", pregunta2)]
})

print("Usuario:", pregunta2)
print("Agente:", resultado2["messages"][-1].content)

Usuario: ¿Qué métodos de pago aceptan en TechShop?
Agente: Aceptamos tarjetas de crédito y débito (Visa, Mastercard, American Express), PayPal, transferencia bancaria y criptomonedas (Bitcoin, Ethereum, USDT).


In [54]:
pregunta3 = "¿Cuánto cuesta el envío express y cuánto tarda?"

resultado3 = agente.invoke({
    "messages": [("user", pregunta3)]
})

print("Usuario:", pregunta3)
print("Agente:", resultado3["messages"][-1].content)

Usuario: ¿Cuánto cuesta el envío express y cuánto tarda?
Agente: El costo del envío express es de $12.99 USD y el tiempo de entrega es de 1 a 3 días hábiles.


In [55]:
pregunta5 = "¿Cuál es la capital de Francia?"

resultado5 = agente.invoke({
    "messages": [("user", pregunta5)]
})

print("Usuario:", pregunta5)
print("Agente:", resultado5["messages"][-1].content)

Usuario: ¿Cuál es la capital de Francia?
Agente: Lo siento, pero la información solicitada no se encuentra en el documento proporcionado. La capital de Francia no se menciona en el texto.
